In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm

data = pd.read_csv("data_wrangling_part2.csv")

In [2]:
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161


This week is data modeling but for my project theres a few things I have to do: Come up with the best way to scrape the test from from the articles, analyze how many articles were lost in the process and redo the encoding for publisher size and checking the minimum article cutoff. Additionally I need to apply FinBert to get the sentiment analysis for the text of each article. Using the text instead of the headline makes sense because it is more representative of what the author is actually saying even at the cost of losing half of the data set due to paywalls and old articles not existing any more. Lastly, I need to determine the best model, how im splitting to handle time as well as being able to classify articles, and how im going to use the model to assign all the publishers a score. 

I used AI to help me make my article downloads (get_text_from_article function from last week) faster with using concurrent features and threading so it does not take 3 days to add the article text colummn. I am going to scrape all the articles, analyze how many rows actually have text, and go from there.  Still have to figure out why I get blocked sometimes from scraping.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_text_from_article(url):
    try:
        article = Article(url)
        article.download()  
        article.parse()
        text = article.text
        return text if len(text) >= 1 else ""
    except Exception:
        return ""

results = [""] * len(data)

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {
        executor.submit(get_text_from_article, url): i
        for i, url in enumerate(data["url"])
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        results[idx] = future.result()

data["article_text"] = results

  0%|          | 45/27357 [00:04<49:44,  9.15it/s]  


In [6]:
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840,
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840,
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814,
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400,
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161,


In [15]:
data.to_csv('temp_scrape.csv', index=False) # Dont want to wait a 3rd time for this scrape once it actually works

In [14]:
scraped_articles = data['article_text'].fillna("").astype(str).str.strip().ne("").sum()
print(scraped_articles)

218


Scraping did not work at all so trying a different library

In [2]:
import requests
import trafilatura
from concurrent.futures import ThreadPoolExecutor, as_completed
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

def get_article_text(url):
    if pd.isna(url) or not str(url).strip():
        return ""

    try:
        response = requests.get(str(url).strip(), headers=headers, timeout=15)
        response.raise_for_status()

        text = trafilatura.extract(
            response.text,
            favor_precision=True,
            include_comments=False,
            include_tables=False
        )

        return text.strip() if text else ""

    except Exception:
        return ""


In [5]:
results = [""] * len(data)

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {
        executor.submit(get_article_text, url): i
        for i, url in enumerate(data["url"])
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        results[idx] = future.result()

data["article_text"] = results

100%|██████████| 27357/27357 [52:12<00:00,  8.73it/s]


In [8]:
data.to_csv('temp_scrape_resume.csv', index=False) # My wifi cut out at 83% 

This scraping is going to take a while so im leaving my notes here: I need to check scrape performance, if its fine then remove all the rows that are empty, reanalyze the author threshold + encoding, FinbertAnalysis, then test a simple model.

In [7]:
# Check performance
scraped_articles = data['article_text'].fillna("").astype(str).str.strip().ne("").sum()
print(scraped_articles)

3817


In [4]:
data = pd.read_csv("temp_scrape_resume.csv")

In [ ]:
# Repeat again a few times ill run this in the night to check for failed articles and to try again
from concurrent.futures import ThreadPoolExecutor, as_completed
data = pd.read_csv("temp_scrape_resume.csv")
if "article_text" not in data.columns:
    data["article_text"] = ""

mask_missing = data["article_text"].fillna("").astype(str).str.strip().eq("")
missing_idx = data.index[mask_missing].tolist()

print("Rows still missing:", len(missing_idx))
print("Rows already scraped:", (~mask_missing).sum())

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(get_article_text, data.at[idx, "url"]): idx
        for idx in missing_idx
    }

    completed = 0
    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        text = future.result()

        if isinstance(text, str) and text.strip() != "":
            data.at[idx, "article_text"] = text

        completed += 1

        if completed % 500 == 0:
            data.to_csv("temp_scrape_resume.csv", index=False)

data.to_csv("temp_scrape_resume.csv", index=False)

Rows still missing: 6055
Rows already scraped: 21302


 50%|████▉     | 3002/6055 [26:29<26:56,  1.89it/s]  


In [5]:
# Check performance
scraped_articles = data['article_text'].fillna("").astype(str).str.strip().ne("").sum()
print(scraped_articles)

23908


After 2 straight days of scraping in increments to avoid ip bans 23908 was the amount of articles I could obtain. Theoretically there are probably more I could scrape but for the sake of this milestone I am going to work with this data

In [6]:
# Remove Rows without text
data["article_text"] = data["article_text"].fillna("").astype(str)

non_empty = data["article_text"].str.strip().ne("").sum()
empty = data["article_text"].str.strip().eq("").sum()

print("Total rows:", len(data))
print("Rows with article text:", non_empty)
print("Rows without text:", empty)

Total rows: 27357
Rows with article text: 23908
Rows without text: 3449


In [7]:
len(data)

27357

Remove the rows I couldnt scrape

In [8]:
data = data[data["article_text"].str.strip().ne("")].copy()

In [9]:
len(data)

23908

In [13]:
data.tail(10)

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text
27347,Option Alert: Heavy Volume in Yum! Brands Janu...,https://www.benzinga.com/markets/options/12/01...,Charles Gross,2012-01-10,YUM,0,2889,2,0.036596,0.087178,0.060439,Never miss a trade again with the fastest news...
27348,Deutsche Bank Raises PT on Yum! Brands to $62,https://www.benzinga.com/analyst-ratings/analy...,Joe Young,2011-12-09,YUM,0,401,0,-0.004486,0.028641,0.124388,Deutsche Bank has published a research report ...
27349,Bernstein Upgrades Yum! Brands to Outperform,https://www.benzinga.com/analyst-ratings/upgra...,Joe Young,2011-12-09,YUM,0,401,0,-0.004486,0.028641,0.124388,Never miss a trade again with the fastest news...
27350,"Goldman Sachs Reiterates Sell, $53 Target on Y...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-08,YUM,0,99,0,0.000174,0.037050,0.171840,Goldman Sachs maintains its Sell rating and $5...
27351,"Bank of America Maintains Buy, $67 Target on Y...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-08,YUM,0,99,0,0.000174,0.037050,0.171840,Bank of America reiterates its Buy rating and ...
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840,"According to J.P. Morgan, Yum Brands (NYSE:\nY..."
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840,© 2026 Benzinga.com. Benzinga does not provide...
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814,Never miss a trade again with the fastest news...
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400,Jefferies maintains its Hold rating and $55 ta...
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161,"Yum! Brands Inc. (NYSE:\nYUM), in advance of i..."


In [14]:
data.to_csv('removed_articles_didnt_scrape.csv', index=False)

In [15]:
data["publisher"].value_counts().describe() # Check publisher counts

count      80.000000
mean      298.850000
std       599.599679
min         3.000000
25%        35.000000
50%        69.000000
75%       235.000000
max      3615.000000
Name: count, dtype: float64

In [16]:
publisher_amount = data['publisher'].value_counts()
publisher_cutoff = publisher_amount[publisher_amount >= 25].index
data = data[data['publisher'].isin(publisher_cutoff)].copy()
len(data)

23837

In [17]:
data['total_article_count'].describe()  # Rencode based on new values

count    23837.000000
mean      1768.651676
std       1387.097502
min         29.000000
25%        429.000000
50%       1314.000000
75%       2428.000000
max       4244.000000
Name: total_article_count, dtype: float64

In [18]:
def encode_article_counts(publisher):
    if publisher <= 429:
        return 0
    elif publisher <= 2428:
        return 1
    else:
        return 2
data['publisher_size'] = data['total_article_count'].apply(encode_article_counts)
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text
27351,"Bank of America Maintains Buy, $67 Target on Y...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-08,YUM,0,99,0,0.000174,0.037050,0.171840,Bank of America reiterates its Buy rating and ...
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840,© 2026 Benzinga.com. Benzinga does not provide...
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814,Never miss a trade again with the fastest news...
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400,Jefferies maintains its Hold rating and $55 ta...
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161,"Yum! Brands Inc. (NYSE:\nYUM), in advance of i..."


In [21]:
# Finbert
data = pd.read_csv('removed_articles_didnt_scrape.csv')
from transformers import pipeline

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)

texts = data["article_text"].fillna("").astype(str).tolist()

prediction_list = []
batch_size = 16

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    preds = finbert(batch, truncation=True)
    prediction_list.extend(preds)

data["finbert_label"] = [prediction["label"] for prediction in prediction_list]
data["finbert_score"] = [prediction["score"] for prediction in prediction_list]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1495/1495 [30:54<00:00,  1.24s/it]


In [22]:
encode_sentiment = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

data["sentiment_encoded"] = data["finbert_label"].map(encode_sentiment)
data.to_csv('sentiment_analysis.csv', index = False)

In [ ]:
print(data["finbert_label"].value_counts())
print(data[["finbert_label", "finbert_score"]].tail())

finbert_label
neutral     19115
negative     2901
positive     1892
Name: count, dtype: int64
      finbert_label  finbert_score
23903       neutral       0.795712
23904       neutral       0.939211
23905       neutral       0.932860
23906      positive       0.717313
23907      positive       0.948789


In [25]:
# Simple model
data.head()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text,finbert_label,finbert_score,sentiment_encoded
0,"UBS Maintains Buy on Adobe, Raises Price Targe...",https://www.benzinga.com/news/20/06/16202690/u...,Benzinga Newsdesk,2020-06-08,ADBE,0,2428,1,0.008950,0.150586,0.237493,Never miss a trade again with the fastest news...,neutral,0.932860,0
1,Shares of several technology companies are tra...,https://www.benzinga.com/wiim/20/05/16075931/s...,Benzinga Newsdesk,2020-05-20,ADBE,0,94,0,-0.022052,0.115684,0.197690,Never miss a trade again with the fastest news...,neutral,0.932860,0
2,"Benzinga's Top Upgrades, Downgrades For May 14...",https://www.benzinga.com/markets/penny-stocks/...,Lisa Levin,2020-05-14,ADBE,0,2321,1,0.075354,0.129295,0.301612,Upgrades\nDowngrades\nInitiations\n© 2026 Benz...,neutral,0.932666,0
3,"DZ Bank Downgrades Adobe to Hold, Announces $3...",https://www.benzinga.com/news/20/05/16029565/d...,Vick Meyer,2020-05-14,ADBE,0,1968,1,0.075354,0.129295,0.301612,Never miss a trade again with the fastest news...,neutral,0.932860,0
4,"BMO Capital Maintains Outperform on Adobe, Rai...",https://www.benzinga.com/news/20/04/15921434/b...,Vick Meyer,2020-04-30,ADBE,0,1968,1,0.037156,0.101911,0.303897,Never miss a trade again with the fastest news...,neutral,0.932860,0


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

Point of modeling is to see if sentiment and other features I made can predict returns and then use this information to find publisher reliability

In [27]:
data = pd.read_csv("sentiment_analysis.csv")

useful_features = [
    "sentiment_encoded",
    "finbert_score",
    "Headline_Class_Prediction",
    "publisher_size",
    "total_article_count"
]

X = data[useful_features]
t = data["return_7d"]

x_train, x_test, t_train, t_test = train_test_split(
    X, t, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(x_train, t_train)

y_pred = model.predict(x_test)

rmse = mean_squared_error(t_test, y_pred) ** 0.5
r2 = r2_score(t_test, y_pred)

print("RMSE:", rmse)
print("R^2:", r2)

RMSE: 0.0557667021490882
R^2: -0.00029076968030739003


Next week I will look into different types of models, hyperparameter tuning, cross validation, and regularization to get a really good model. This week was more of a test model and make sure the data is in a good spot because scraping was really difficult.

Linear Regression predicting the return is not a good approach to evaluating publishers because I do not want to have to predict returns to evaluate publishers. All I want to do is classify publishers based on their sentiment vs. the stock performance in intervals.